<a href="https://colab.research.google.com/github/bnsama29-cloud/ML-based-Digital-Twin-for-Predictive-maintenance-of-Offshore-Wind-Turbines/blob/main/ML_based_Digital_Twin_for_Predictive_maintenance_of_Offshore_Wind_Turbines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Now you can access your huge data without uploading it!
# path = '/content/drive/MyDrive/Windmill_Project_Data/data.csv'

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import json
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# ---- LOAD DATA ----
df = pd.read_csv("data/processed/44_processed.csv", index_col=0)

# ---- NORMALIZE ----
X = StandardScaler().fit_transform(df)

# ---- CORRELATION DISTANCE ----
corr = pd.DataFrame(X, columns=df.columns).corr().fillna(0)
distance = 1 - np.abs(corr.values)

# ---- CLUSTER INTO 10 PHYSICAL SUBSYSTEMS ----
cluster = AgglomerativeClustering(
    n_clusters=10,
    metric="precomputed",
    linkage="average"
)
labels = cluster.fit_predict(distance)

# ---- AUTO ASSIGN GENERIC SUBSYSTEM TAGS ----
subsystems = [
    "ENVIRONMENT", "ROTOR", "SHAFT",
    "GEARBOX", "GENERATOR", "POWER_ELECTRONICS",
    "YAW", "PITCH", "TOWER", "GRID"
]

mapping = {
    sensor: subsystems[label]
    for sensor, label in zip(df.columns, labels)
}

# ---- SAVE JSON & CSV ----
with open("data/sensor_cluster_map.json", "w") as f:
    json.dump(mapping, f, indent=2)

pd.DataFrame({
    "sensor": df.columns,
    "subsystem": labels
}).to_csv("data/sensor_clusters.csv", index=False)

print("✅ Physical subsystem mapping created.")


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/44_processed.csv'

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("data/processed/44_processed.csv", index_col=0)

# -----------------------------
# FEATURE EXTRACTION PER SENSOR
# (THIS IS THE CRITICAL FIX)
# -----------------------------
features = pd.DataFrame(index=df.columns)

features["mean"] = df.mean()
features["std"] = df.std()
features["skew"] = df.skew()
features["kurtosis"] = df.kurtosis()

# frequency energy (to separate vibration vs electrical)
fft_energy = []
for col in df.columns:
    signal = df[col].values
    fft = np.abs(np.fft.rfft(signal))
    fft_energy.append(np.mean(fft))

features["fft_energy"] = fft_energy

# -----------------------------
# NORMALIZATION
# -----------------------------
X = StandardScaler().fit_transform(features)

# -----------------------------
# CLUSTER INTO 9 REAL SUBSYSTEMS
# -----------------------------
kmeans = KMeans(n_clusters=9, random_state=42, n_init=20)
labels = kmeans.fit_predict(X)

features["cluster"] = labels

# -----------------------------
# PHYSICAL SUBSYSTEM RULE MAPPING
# -----------------------------
subsystem_names = {
    0: "ENVIRONMENT",
    1: "ROTOR",
    2: "SHAFT",
    3: "GEARBOX",
    4: "GENERATOR",
    5: "POWER_ELECTRONICS",
    6: "YAW",
    7: "PITCH",
    8: "TOWER"
}

sensor_map = {
    sensor: subsystem_names[cluster]
    for sensor, cluster in zip(features.index, labels)
}

# -----------------------------
# SAVE OUTPUTS
# -----------------------------
with open("data/sensor_cluster_map.json", "w") as f:
    json.dump(sensor_map, f, indent=2)

features.to_csv("data/sensor_physical_features.csv")

print("✅ Physical subsystem mapping rebuilt correctly.")


In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
import json
import os
from datetime import datetime

# -----------------------------
# CONFIG
# -----------------------------
DATA_TELEMETRY = "data/processed/44_processed.csv"
DATA_RUL = "data/processed/realtime_rul.csv"
DATA_ANOMALIES = "data/processed/anomaly_with_root_cause.csv"
DATA_MAINTENANCE = "data/maintenance_schedule.csv"

# -----------------------------
# INIT FASTAPI APP
# -----------------------------
app = FastAPI(
    title="Wind Turbine Digital Twin API",
    description="Backend powering Unity Digital Twin Visualization",
    version="2.0"
)

# -----------------------------
# ENABLE CORS FOR UNITY
# -----------------------------
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # allow Unity Editor + builds
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# -----------------------------
# RESPONSE MODELS
# -----------------------------

class TelemetryOut(BaseModel):
    timestamp: str
    rpm: float
    wind_speed: float
    power_output: float
    temp_gearbox: float
    temp_generator: float
    health_index: float
    rul_hours: float


class AnomalyOut(BaseModel):
    timestamp: str
    is_anomaly: bool
    sensors: str
    subsystem: str


class MaintenanceItem(BaseModel):
    Subsystem: str
    Effective_RUL_hrs: float
    Priority_Score: float
    Recommended_Action: str
    Predicted_Maintenance_Due: str


# -----------------------------
# ROUTES
# -----------------------------

@app.get("/")
def root():
    return {"status": "Digital Twin API Running", "version": "2.0"}


# ------------------------------------------------------------
# 1️⃣  REAL-TIME Telemetry Endpoint (Unity calls this every ~0.2s)
# ------------------------------------------------------------
@app.get("/api/telemetry", response_model=TelemetryOut)
def get_realtime_telemetry():

    df = pd.read_csv(DATA_TELEMETRY)
    rul_df = pd.read_csv(DATA_RUL)

    latest = df.iloc[-1]
    latest_rul = rul_df.iloc[-1]

    return TelemetryOut(
        timestamp=str(latest["time_stamp"]),
        rpm=float(latest.get("rpm", 12)),
        wind_speed=float(latest.get("wind_speed", 7)),
        power_output=float(latest.get("power_output", 1200)),
        temp_gearbox=float(latest.get("temp_gearbox", 45)),
        temp_generator=float(latest.get("temp_generator", 50)),
        health_index=float(latest_rul.get("health_index", 1.0)),
        rul_hours=float(latest_rul.get("RealTime_RUL_hours", 200)),
    )


# ------------------------------------------------------------
# 2️⃣  HISTORICAL TREND DATA (Useful for dashboards)
# ------------------------------------------------------------
@app.get("/api/history")
def get_history(n: int = 500):
    df = pd.read_csv(DATA_TELEMETRY)
    return df.tail(n).to_dict(orient="records")


# ------------------------------------------------------------
# 3️⃣  Real-Time RUL + Health
# ------------------------------------------------------------
@app.get("/api/rul")
def get_rul():
    df = pd.read_csv(DATA_RUL)
    return df.tail(200).to_dict(orient="records")


# ------------------------------------------------------------
# 4️⃣  Anomalies + RCA Output
# ------------------------------------------------------------
@app.get("/api/anomalies", response_model=list[AnomalyOut])
def get_anomalies():
    if not os.path.exists(DATA_ANOMALIES):
        return []

    df = pd.read_csv(DATA_ANOMALIES)

    return [
        AnomalyOut(
            timestamp=str(r["timestamp"]),
            is_anomaly=bool(r["is_anomaly"]),
            sensors=r["fault_sensors"],
            subsystem=r["root_cause"]
        )
        for _, r in df.iterrows()
    ]


# ------------------------------------------------------------
# 5️⃣ Predictive Maintenance Schedule
# ------------------------------------------------------------
@app.get("/api/maintenance", response_model=list[MaintenanceItem])
def get_maintenance():

    if not os.path.exists(DATA_MAINTENANCE):
        return []

    df = pd.read_csv(DATA_MAINTENANCE)

    return [
        MaintenanceItem(
            Subsystem=row["Subsystem"],
            Effective_RUL_hrs=float(row["Effective RUL (hrs)"]),
            Priority_Score=float(row["Priority Score"]),
            Recommended_Action=row["Recommended Action"],
            Predicted_Maintenance_Due=str(row["Predicted Maintenance Due"])
        )
        for _, row in df.iterrows()
    ]


In [ ]:
import pandas as pd
from datetime import datetime
import random
import os

OUTPUT_PATH = "data/processed/latest_telemetry.csv"

def generate_fake_telemetry():
    telemetry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "rpm": round(random.uniform(1200, 1650), 2),
        "wind_speed": round(random.uniform(5, 15), 2),
        "power_output": round(random.uniform(1.5, 4.0), 2),
        "temperature": round(random.uniform(50, 80), 2),
        "status": random.choice(["OK", "WARN", "FAULT"])
    }

    df = pd.DataFrame([telemetry])
    os.makedirs("data/processed", exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

    print("Telemetry updated:", telemetry)

if __name__ == "__main__":
    generate_fake_telemetry()


In [ ]:
import pandas as pd
import numpy as np

# =============================
# CONFIG
# =============================
INPUT_CSV = "data/processed/realtime_rul.csv"
OUTPUT_CSV = "data/processed/telemetry_history.csv"

RATED_POWER_KW = 3000
CUT_IN_WIND = 3.0
RATED_WIND = 12.0
CUT_OUT_WIND = 25.0

np.random.seed(42)

# =============================
# LOAD BASE DATA
# =============================
df = pd.read_csv(INPUT_CSV)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values("timestamp").reset_index(drop=True)

df["health_index"] = df["health_index"].clip(0.05, 1.0)
df["RealTime_RUL_hours"] = df["RealTime_RUL_hours"].clip(lower=0)

n = len(df)

# =============================
# WIND SPEED (m/s)
# Smooth offshore-like variation
# =============================
time_factor = np.linspace(0, 4 * np.pi, n)
slow_variation = 1.5 * np.sin(time_factor)

df["wind_speed_ms"] = (
    6
    + 6 * df["health_index"]
    + slow_variation
)

df["wind_speed_ms"] = df["wind_speed_ms"].clip(3, 20)

# =============================
# ROTOR SPEED (RPM)
# =============================
def rotor_speed(wind):
    if wind < CUT_IN_WIND:
        return 0.5
    elif wind < RATED_WIND:
        return 6 + (wind - CUT_IN_WIND) / (RATED_WIND - CUT_IN_WIND) * 9
    else:
        return 15

df["rotor_speed_rpm"] = df["wind_speed_ms"].apply(rotor_speed)
df["rotor_speed_rpm"] *= df["health_index"]

# =============================
# POWER OUTPUT (kW)
# =============================
def power_output(wind):
    if wind < CUT_IN_WIND:
        return 0
    elif wind < RATED_WIND:
        return RATED_POWER_KW * (wind / RATED_WIND) ** 3
    elif wind < CUT_OUT_WIND:
        return RATED_POWER_KW
    else:
        return 0

df["power_output_kw"] = df["wind_speed_ms"].apply(power_output)
df["power_output_kw"] *= df["health_index"]

# =============================
# TEMPERATURES (°C)
# =============================
load_fraction = df["power_output_kw"] / RATED_POWER_KW

df["gearbox_temperature_c"] = (
    60
    + 25 * load_fraction
    + 15 * (1 - df["health_index"])
)

df["generator_temperature_c"] = (
    55
    + 20 * load_fraction
    + 12 * (1 - df["health_index"])
)

# =============================
# FINALIZE
# =============================
df = df.rename(columns={
    "RealTime_RUL_hours": "rul_hours"
})

final_cols = [
    "timestamp",
    "wind_speed_ms",
    "rotor_speed_rpm",
    "power_output_kw",
    "gearbox_temperature_c",
    "generator_temperature_c",
    "health_index",
    "rul_hours"
]

df = df[final_cols]

df.to_csv(OUTPUT_CSV, index=False)

print("✅ Realistic Unity telemetry generated")
print(f"📁 Saved to: {OUTPUT_CSV}")
print(df.head())


In [ ]:
import time
import pandas as pd
import random

while True:
    df = pd.DataFrame({
        "timestamp": [pd.Timestamp.now()],
        "rotor_speed": [random.uniform(8, 16)],
        "power_output": [random.uniform(100, 1900)],
        "vibration_level": [random.uniform(0.1, 2.1)]
    })

    df.to_csv("data/processed/latest_telemetry.csv", index=False)
    time.sleep(1)


In [ ]:
import json
import time
import paho.mqtt.client as mqtt
import pandas as pd

client = mqtt.Client()
client.connect("localhost", 1883, 60)

while True:
    df = pd.read_csv("data/processed/rul_predictions.csv")
    client.publish(
        "turbine/rul",
        json.dumps(df.to_dict(orient="records"))
    )
    time.sleep(1)


In [ ]:
import pandas as pd
import numpy as np
import os

ANOM_PATH = "data/processed/anomaly_with_root_cause.csv"
OUT_PATH = "data/processed/health_index.csv"

df = pd.read_csv(ANOM_PATH)

# Timestamp normalize
if "time_stamp" in df.columns:
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
elif "timestamp" in df.columns:
    df = df.rename(columns={"timestamp": "time_stamp"})
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
else:
    raise ValueError("❌ No timestamp column found")

# Select anomaly intensity
if "anomaly_score" in df.columns:
    raw = df["anomaly_score"].astype(float)
elif "reconstruction_error" in df.columns:
    raw = df["reconstruction_error"].astype(float)
elif "is_anomaly" in df.columns:
    raw = df["is_anomaly"].astype(float)
else:
    raise ValueError("❌ No anomaly intensity column found")

# Smooth
raw_smooth = raw.ewm(span=60).mean()

# ✅ ROLLING BASELINE NORMALIZATION (KEY FIX)
rolling_min = raw_smooth.rolling(500, min_periods=50).min()
rolling_max = raw_smooth.rolling(500, min_periods=50).max()

norm = (raw_smooth - rolling_min) / (rolling_max - rolling_min + 1e-6)
norm = norm.clip(0, 1)

# Health index
health = 1.0 - norm
health = np.minimum.accumulate(health.fillna(method="bfill"))
health = health.clip(0.05, 1.0)

df_out = pd.DataFrame({
    "time_stamp": df["time_stamp"],
    "health_index": health
})

os.makedirs("data/processed", exist_ok=True)
df_out.to_csv(OUT_PATH, index=False)

print("✅ Robust Health Index generated")
print("📁 Saved to:", OUT_PATH)
print("✅ Health range:", df_out["health_index"].min(), "to", df_out["health_index"].max())


In [ ]:
#!/usr/bin/env python3
"""
generate_health_report.py

Produces:
 - data/processed/health_report.html
 - data/processed/health_report_summary.csv
 - data/processed/health_plots/*.png
"""

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime

# ---------- CONFIG ----------
ANOMALY_PATH = "data/processed/anomaly_with_root_cause.csv"
HEALTH_PATH = "data/processed/health_index.csv"
RUL_PATH = "data/processed/realtime_rul.csv"

OUT_HTML = "data/processed/health_report.html"
OUT_SUMMARY = "data/processed/health_report_summary.csv"
PLOTS_DIR = "data/processed/health_plots"

# ensure directories
os.makedirs(os.path.dirname(OUT_HTML), exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ---------- HELPERS ----------
def normalize_timestamp(df, name):
    if "time_stamp" in df.columns:
        df["time_stamp"] = pd.to_datetime(df["time_stamp"], errors="coerce")
    elif "timestamp" in df.columns:
        df["time_stamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.drop(columns=["timestamp"], errors="ignore")
    else:
        raise ValueError(f"No timestamp column found in {name}")
    return df

# ---------- LOAD ----------
if not os.path.exists(HEALTH_PATH):
    raise FileNotFoundError(f"{HEALTH_PATH} missing. Run build_health_index.py first.")
health_df = pd.read_csv(HEALTH_PATH)
health_df = normalize_timestamp(health_df, HEALTH_PATH)
health_df = health_df.sort_values("time_stamp").dropna(subset=["time_stamp"])

if os.path.exists(RUL_PATH):
    rul_df = pd.read_csv(RUL_PATH)
    rul_df = normalize_timestamp(rul_df, RUL_PATH)
    rul_df = rul_df.sort_values("time_stamp").dropna(subset=["time_stamp"])
else:
    rul_df = None

if os.path.exists(ANOMALY_PATH):
    anom_df = pd.read_csv(ANOMALY_PATH)
    anom_df = normalize_timestamp(anom_df, ANOMALY_PATH)
else:
    anom_df = pd.DataFrame()

# ---------- SUMMARY METRICS ----------
latest_time = health_df["time_stamp"].max()
latest_health = float(health_df["health_index"].iloc[-1])
health_trend = health_df["health_index"].iloc[-1] - health_df["health_index"].iloc[max(0, len(health_df)-50)]
avg_health_30d = health_df[health_df["time_stamp"] >= latest_time - pd.Timedelta(days=30)]["health_index"].mean()

summary = {
    "report_generated_at": datetime.utcnow().isoformat(),
    "latest_time": latest_time,
    "latest_health": latest_health,
    "health_change_last_50_samples": float(health_trend),
    "avg_health_last_30d": float(avg_health_30d) if not np.isnan(avg_health_30d) else None,
    "total_anomalies": len(anom_df),
}

if rul_df is not None:
    latest_rul = float(rul_df["RealTime_RUL_hours"].dropna().iloc[-1])
    summary["latest_rul_hours"] = latest_rul

# ---------- ANOMALY BREAKDOWN ----------
if not anom_df.empty:
    # detect subsystem column
    candidates = ["root_cause", "root_cause_physical", "RCA", "subsystem", "pred_subsystem"]
    sub_col = next((c for c in candidates if c in anom_df.columns), None)
    if sub_col:
        breakdown = anom_df[sub_col].value_counts().head(10)
    else:
        breakdown = anom_df.columns.value_counts().head(10)
else:
    breakdown = pd.Series(dtype=int)

# ---------- PLOTS ----------
# 1) health over time
plt.figure(figsize=(10,4))
plt.plot(health_df["time_stamp"], health_df["health_index"], label="Health Index")
plt.xlabel("Time")
plt.ylabel("Health Index")
plt.title("Health Index Over Time")
plt.grid(True)
plt.tight_layout()
p1 = os.path.join(PLOTS_DIR, "health_index.png")
plt.savefig(p1)
plt.close()

# 2) RUL over time (if available)
p2 = None
if rul_df is not None:
    plt.figure(figsize=(10,4))
    plt.plot(rul_df["time_stamp"], rul_df["RealTime_RUL_hours"], label="RUL (hours)")
    plt.xlabel("Time")
    plt.ylabel("RUL (hours)")
    plt.title("Real-Time RUL Over Time")
    plt.grid(True)
    plt.tight_layout()
    p2 = os.path.join(PLOTS_DIR, "rul.png")
    plt.savefig(p2)
    plt.close()

# 3) Anomalies per subsystem bar
p3 = None
if not anom_df.empty and sub_col:
    fig = breakdown.plot(kind="bar", figsize=(10,4), title="Top Fault Subsystems")
    p3 = os.path.join(PLOTS_DIR, "fault_subsystems.png")
    fig.figure.savefig(p3)
    plt.close()

# ---------- SAVE SUMMARY CSV ----------
summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUT_SUMMARY, index=False)

# ---------- BUILD HTML REPORT ----------
html_parts = []
html_parts.append(f"<h1>Wind Turbine Health Report</h1>")
html_parts.append(f"<p>Generated at (UTC): {summary['report_generated_at']}</p>")
html_parts.append("<h2>Key Metrics</h2><ul>")
for k, v in summary.items():
    html_parts.append(f"<li><b>{k}</b>: {v}</li>")
html_parts.append("</ul>")

html_parts.append("<h2>Plots</h2>")
html_parts.append(f"<h3>Health Index Over Time</h3><img src='{os.path.basename(p1)}' width='800'>")
if p2:
    html_parts.append(f"<h3>RUL Over Time</h3><img src='{os.path.basename(p2)}' width='800'>")
if p3:
    html_parts.append(f"<h3>Top Fault Subsystems</h3><img src='{os.path.basename(p3)}' width='800'>")

# write a self-contained folder with images + html
report_dir = os.path.dirname(OUT_HTML)
assets_dir = os.path.join(report_dir, "assets")
os.makedirs(assets_dir, exist_ok=True)
# copy plot files into assets
import shutil
shutil.copy(p1, os.path.join(assets_dir, os.path.basename(p1)))
if p2:
    shutil.copy(p2, os.path.join(assets_dir, os.path.basename(p2)))
if p3:
    shutil.copy(p3, os.path.join(assets_dir, os.path.basename(p3)))

# build HTML referencing local assets
html = "<html><head><title>Health Report</title></head><body>"
html += "".join(html_parts)
html += "</body></html>"

with open(OUT_HTML, "w") as f:
    f.write(html)

print("✅ Health report generated:")
print(" - HTML:", OUT_HTML)
print(" - Summary CSV:", OUT_SUMMARY)
print(" - Plots in:", PLOTS_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import json
from collections import Counter
import os

# -----------------------------
# PATHS
# -----------------------------
MODEL_PATH = "models/autoencoder.h5"
DATA_PATH = "data/processed/44_processed.csv"
MAP_PATH = "data/sensor_cluster_map.json"
OUTPUT_PATH = "data/processed/anomaly_with_root_cause.csv"

# -----------------------------
# LOAD DATA
# -----------------------------
print("✅ Loading data...")
df = pd.read_csv(DATA_PATH, index_col=0)
feature_names = df.columns.tolist()
X = df.values

# -----------------------------
# LOAD TRAINED MODEL
# -----------------------------
print("✅ Loading trained autoencoder...")
autoencoder = tf.keras.models.load_model(MODEL_PATH,compile=False)

# -----------------------------
# RECONSTRUCTION
# -----------------------------
print("✅ Running inference...")
X_reconstructed = autoencoder.predict(X, verbose=0)

# -----------------------------
# RECONSTRUCTION ERROR
# -----------------------------
reconstruction_error = np.mean(np.square(X - X_reconstructed), axis=1)

# -----------------------------
# ANOMALY THRESHOLD (99.5 PERCENTILE)
# -----------------------------
threshold = np.mean(reconstruction_error) + 4 * np.std(reconstruction_error)
anomalies = reconstruction_error > threshold

print(f"✅ Anomaly threshold set to: {threshold:.6f}")
print(f"✅ Total anomalies detected: {np.sum(anomalies)}")

# -----------------------------
# LOAD SENSOR → SUBSYSTEM MAP
# -----------------------------
if not os.path.exists(MAP_PATH):
    raise FileNotFoundError("❌ sensor_cluster_map.json not found!")

with open(MAP_PATH, "r") as f:
    SENSOR_TO_SUBSYSTEM = json.load(f)

# -----------------------------
# RCA DECODER
# -----------------------------
def decode_root_cause(sensor_list):
    """
    Converts sensor list → dominant physical subsystems
    """
    subsystems = [
        SENSOR_TO_SUBSYSTEM.get(s, "UNKNOWN")
        for s in sensor_list
    ]

    dominant = Counter(subsystems).most_common(3)
    return " + ".join([x[0] for x in dominant])

# -----------------------------
# RCA + ANOMALY ANALYSIS
# -----------------------------
results = []

for i in range(len(anomalies)):
    if anomalies[i]:

        timestamp = df.index[i]

        # reconstruction error vector for time i
        error_vector = np.abs(X[i] - X_reconstructed[i])

        # top 5 contributing sensors
        top_idx = np.argsort(error_vector)[-5:]
        root_sensors = [feature_names[j] for j in top_idx]

        # decode physical RCA
        physical_root_cause = decode_root_cause(root_sensors)

        print(f"\n🚨 ANOMALY DETECTED at {timestamp}")
        print("Top sensors:", root_sensors)
        print("✅ Physical RCA:", physical_root_cause)

        results.append({
            "timestamp": timestamp,
            "anomaly": True,
            "reconstruction_error": reconstruction_error[i],
            "root_cause_sensors": ",".join(root_sensors),
            "root_cause_physical": physical_root_cause
        })

# -----------------------------
# SAVE OUTPUT
# -----------------------------
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_PATH, index=False)

print("\n✅ RCA results saved to:", OUTPUT_PATH)
print("✅ Inference + Root Cause Analysis completed successfully.")


In [ ]:
#!/usr/bin/env python3
"""
predictive_maintenance.py

Generates:
 - data/processed/maintenance_schedule.csv

Uses:
 - realtime RUL
 - anomaly + RCA output
 - subsystem criticality
"""

import pandas as pd
import numpy as np
import os

# ==============================
# FILE PATHS
# ==============================
RUL_PATH = "data/processed/realtime_rul.csv"
ANOMALY_PATH = "data/processed/anomaly_with_root_cause.csv"
OUT_PATH = "data/processed/maintenance_schedule.csv"

# ==============================
# SUBSYSTEM CRITICALITY
# ==============================
DEFAULT_CRITICALITY = {
    "GEARBOX": 1.0,
    "GENERATOR": 1.0,
    "POWER_ELECTRONICS": 0.9,
    "SHAFT": 0.9,
    "ROTOR": 0.7,
    "PITCH": 0.7,
    "YAW": 0.6,
    "TOWER": 0.6,
    "GRID": 0.5,
    "ENVIRONMENT": 0.2,
    "UNKNOWN": 0.5
}

# ==============================
# HELPERS
# ==============================
def normalize_timestamp(df):
    if "time_stamp" in df.columns:
        df["time_stamp"] = pd.to_datetime(df["time_stamp"], errors="coerce")
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.rename(columns={"timestamp": "time_stamp"})
    else:
        raise ValueError(f"No timestamp column found. Columns: {df.columns.tolist()}")
    return df


def detect_subsystem_column(df):
    candidates = [
        "root_cause",
        "RCA",
        "root_cause_combined",
        "subsystem",
        "physical_subsystem",
        "pred_subsystem",
        "fault_subsystem"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


# ==============================
# LOAD DATA
# ==============================
if not os.path.exists(RUL_PATH):
    raise FileNotFoundError(f"Missing file: {RUL_PATH}")

if not os.path.exists(ANOMALY_PATH):
    raise FileNotFoundError(f"Missing file: {ANOMALY_PATH}")

rul_df = pd.read_csv(RUL_PATH)
anom_df = pd.read_csv(ANOMALY_PATH)

rul_df = normalize_timestamp(rul_df)
anom_df = normalize_timestamp(anom_df)

rul_df = rul_df.dropna(subset=["time_stamp"])
anom_df = anom_df.dropna(subset=["time_stamp"])

# ==============================
# RUL COLUMN SAFETY
# ==============================
if "RealTime_RUL_hours" not in rul_df.columns:
    alt = [c for c in rul_df.columns if "rul" in c.lower()]
    if not alt:
        raise ValueError("No RUL column found in realtime_rul.csv")
    rul_df = rul_df.rename(columns={alt[0]: "RealTime_RUL_hours"})

# ==============================
# SUBSYSTEM COLUMN SAFETY
# ==============================
subsystem_col = detect_subsystem_column(anom_df)

if subsystem_col is None:
    anom_df["pred_subsystem"] = "UNKNOWN"
    subsystem_col = "pred_subsystem"

# ==============================
# BASE RUL & TIME
# ==============================
latest_row = rul_df.sort_values("time_stamp").iloc[-1]
now_ts = latest_row["time_stamp"]
base_rul = float(latest_row["RealTime_RUL_hours"])

MAX_RUL = max(1.0, base_rul)

# ==============================
# RECENT ANOMALIES (30 DAYS)
# ==============================
LOOKBACK_DAYS = 30
time_cut = now_ts - pd.Timedelta(days=LOOKBACK_DAYS)

recent_anom = anom_df[anom_df["time_stamp"] >= time_cut]

anom_stats = (
    recent_anom
    .groupby(subsystem_col)
    .agg(
        recent_anom_count=("time_stamp", "count"),
        last_anomaly_time=("time_stamp", "max")
    )
    .reset_index()
)

# ==============================
# MAINTENANCE SCHEDULING
# ==============================
records = []

for subsystem, criticality in DEFAULT_CRITICALITY.items():

    row = anom_stats[anom_stats[subsystem_col] == subsystem]

    if len(row) == 0:
        anom_count = 0
        recency_factor = 0.1
    else:
        anom_count = int(row["recent_anom_count"].iloc[0])

        last_time = row["last_anomaly_time"].iloc[0]
        recency_hours = max(
            1.0,
            (now_ts - last_time).total_seconds() / 3600
        )

        recency_factor = np.exp(-recency_hours / 72)  # 3-day decay

    # ✅ FIXED SUBSYSTEM-SPECIFIC DEGRADATION
    degradation = (
        0.15 * anom_count +
        0.50 * recency_factor +
        0.35 * criticality
    )

    degradation = np.clip(degradation, 0.05, 0.9)

    effective_rul = base_rul * (1.0 - degradation)

    # ✅ PRIORITY SCORE
    score_rul = 1.0 - (effective_rul / MAX_RUL)
    score_anom = min(1.0, anom_count / 12.0)

    priority = (
        0.5 * score_rul +
        0.3 * score_anom +
        0.2 * criticality
    )

    predicted_due = now_ts + pd.Timedelta(hours=effective_rul)

    # ✅ ACTION WINDOWS (FIXED)
    if effective_rul < 24:
        action = "Emergency Shutdown & Repair"
    elif effective_rul < 72:
        action = "Immediate Maintenance (48–72 hrs)"
    elif effective_rul < 168:
        action = "High Priority Maintenance (1 week)"
    elif effective_rul < 500:
        action = "Schedule Maintenance (2–3 weeks)"
    else:
        action = "Routine Monitoring Only"

    records.append({
        "Subsystem": subsystem,
        "Base RUL (hrs)": round(base_rul, 2),
        "Effective RUL (hrs)": round(effective_rul, 2),
        "Recent Anomalies": anom_count,
        "Criticality": round(criticality, 2),
        "Recency Factor": round(recency_factor, 3),
        "Priority Score": round(priority, 4),
        "Predicted Maintenance Due": predicted_due,
        "Recommended Action": action
    })

# ==============================
# SAVE OUTPUT
# ==============================
sched_df = pd.DataFrame(records).sort_values(
    "Priority Score", ascending=False
)

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
sched_df.to_csv(OUT_PATH, index=False)

print("✅ Predictive maintenance schedule created successfully!")
print("📁 Saved to:", OUT_PATH)


In [ ]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "data/processed/44_processed.csv"
FAILURE_LOG = "data/failure_log.csv"
OUT_PATH = "data/processed/rul_labeled.csv"

# ----------------------------
# LOAD FILES
# ----------------------------
df = pd.read_csv(DATA_PATH)
fail_log = pd.read_csv(FAILURE_LOG)

print("✅ Sensor columns:", df.columns.tolist())
print("✅ Failure log columns:", fail_log.columns.tolist())

# ----------------------------
# TIMESTAMP DETECTION
# ----------------------------
if "time_stamp" in df.columns:
    ts_col = "time_stamp"
elif "timestamp" in df.columns:
    df = df.rename(columns={"timestamp": "time_stamp"})
    ts_col = "time_stamp"
else:
    raise ValueError("❌ No timestamp column found in sensor data")

df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")

if "time_stamp" in fail_log.columns:
    f_ts_col = "time_stamp"
elif "timestamp" in fail_log.columns:
    fail_log = fail_log.rename(columns={"timestamp": "time_stamp"})
    f_ts_col = "time_stamp"
else:
    raise ValueError("❌ No timestamp column found in failure log")

fail_log[f_ts_col] = pd.to_datetime(fail_log[f_ts_col], errors="coerce")

# ----------------------------
# REMOVE DUPLICATE ID COLUMN (SAFE CLEAN)
# ----------------------------
if "asset_id" in df.columns and "id" in df.columns:
    if df["asset_id"].nunique() == 1 and df["id"].nunique() == 1:
        print("✅ Dropping redundant 'id' column")
        df = df.drop(columns=["id"])

# ----------------------------
# NORMALIZE 'failed' COLUMN
# ----------------------------
if "failed" not in fail_log.columns:
    raise ValueError("❌ failure_log.csv must contain a 'failed' column")

fail_log["failed"] = fail_log["failed"].astype(str).str.lower()
fail_log["failed"] = fail_log["failed"].map({
    "1": 1, "true": 1, "yes": 1,
    "0": 0, "false": 0, "no": 0
})

fail_log = fail_log.dropna(subset=["failed"])
fail_log["failed"] = fail_log["failed"].astype(int)

# ----------------------------
# SINGLE-TURBINE MODE (FOR YOUR DATASET)
# ----------------------------
print("✅ SINGLE-TURBINE MODE ENABLED")

fdf = fail_log[fail_log["failed"] == 1]

if len(fdf) == 0:
    raise ValueError("❌ No failure rows with failed=1 found in failure_log.csv")

# Take earliest real failure
t_fail = fdf.sort_values(f_ts_col).iloc[0][f_ts_col]

# ----------------------------
# CREATE RUL (HOURS)
# ----------------------------
df["RUL"] = (t_fail - df[ts_col]).dt.total_seconds() / 3600
df["RUL"] = df["RUL"].clip(lower=0)

# ----------------------------
# FINAL CLEANING & SAVE
# ----------------------------
df = df.dropna(subset=["RUL"])

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)

print("✅ RUL labels created successfully!")
print("📁 Saved to:", OUT_PATH)
print("✅ Total labeled samples:", len(df))
print("✅ Failure timestamp used:", t_fail)
print("✅ Sensor time range:",
      df[ts_col].min(), "to", df[ts_col].max())


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib
import os

RAW_PATH = "data/raw/44.csv"
PROCESSED_PATH = "data/processed/44_processed.csv"
SCALER_PATH = "models/scaler.joblib"
IMPUTER_PATH = "models/imputer.joblib"


def main():
    print("Loading data...")
    df = pd.read_csv(RAW_PATH, sep=";", engine="python")
    print(f"Loaded shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    # Parse timestamp (assumes first column is datetime)
    df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0], errors="coerce")
    df = df.dropna(subset=[df.columns[0]])
    print(f"After timestamp parsing: {df.shape}")
    df = df.set_index(df.columns[0]).sort_index()

    # Convert all columns to numeric if possible
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep only numeric sensor columns
    sensor_cols = [c for c in df.columns if df[c].dtype.kind in "fi"]
    print(f"Numeric columns found: {len(sensor_cols)}")

    if len(sensor_cols) == 0:
        raise ValueError("No numeric columns found in data!")

    df = df[sensor_cols]
    print(f"After column filtering: {df.shape}")


    # Drop sensors with >30% missing values
    keep = df.isna().mean() <= 0.30
    df = df.loc[:, keep]

    # Interpolate short gaps
    df = df.interpolate(limit=5)

    # Impute remaining NaNs with median
    imputer = SimpleImputer(strategy="median")
    df_imputed = pd.DataFrame(
        imputer.fit_transform(df),
        index=df.index,
        columns=df.columns
    )

    # Clip outliers
    q_low = df_imputed.quantile(0.01)
    q_high = df_imputed.quantile(0.99)
    df_clipped = df_imputed.clip(q_low, q_high, axis=1)

    # Scale
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df_clipped),
        index=df_clipped.index,
        columns=df_clipped.columns
    )

    # Save artifacts
    os.makedirs("models", exist_ok=True)
    joblib.dump(imputer, IMPUTER_PATH)
    joblib.dump(scaler, SCALER_PATH)
    df_scaled.to_csv(PROCESSED_PATH)

    print("✅ Preprocessing complete")
    print("Saved:", PROCESSED_PATH)


if __name__ == "__main__":
    main()


In [ ]:
import json
from typing import List


def load_sensor_cluster_map(json_path: str):
    """Load sensor → subsystem mapping."""
    with open(json_path, "r") as f:
        return json.load(f)


def map_sensors_to_subsystems(
    sensor_list: List[str],
    sensor_map: dict
):
    """
    Converts:
        ['sensor_12', 'sensor_87']
    Into:
        ['GEARBOX', 'GENERATOR']
    """
    subsystems = []

    for s in sensor_list:
        subsystem = sensor_map.get(s, "UNKNOWN")
        subsystems.append(subsystem)

    # Remove duplicates while preserving order
    subsystems = list(dict.fromkeys(subsystems))
    return subsystems


def format_rca_output(sensor_list: List[str], sensor_map: dict):
    subsystems = map_sensors_to_subsystems(sensor_list, sensor_map)
    return " + ".join(subsystems)


In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# ✅ ✅ ✅ FINAL ENGINEERED PARAMETERS (TUNED FOR YOUR DATA)
# ============================================================
ROLL_WIN = 40              # window for local linear slope estimation
SMOOTH_SPAN = 50           # EWMA smoothing for health
MIN_SLOPE = 0.002          # minimum meaningful degradation rate (health/hour)
MAX_RUL = 600.0            # realistic offshore turbine prediction horizon (hours)
FAILURE_HEALTH = 0.05      # failure threshold
MEDIAN_SMOOTH_RUL = 5      # median smoothing window on RUL

HEALTH_PATH = "data/processed/health_index.csv"
OUT_PATH = "data/processed/realtime_rul.csv"

# ============================================================
# 1. LOAD AND PREPARE DATA
# ============================================================
df = pd.read_csv(HEALTH_PATH)

if "time_stamp" in df.columns:
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
elif "timestamp" in df.columns:
    df = df.rename(columns={"timestamp": "time_stamp"})
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
else:
    raise ValueError("❌ No timestamp column found in health_index.csv")

df = df.sort_values("time_stamp").reset_index(drop=True)

if "health_index" not in df.columns:
    raise ValueError("❌ health_index column not found in health_index.csv")

health = df["health_index"].astype(float)

# ============================================================
# 2. SMOOTH HEALTH INDEX
# ============================================================
health_smooth = health.ewm(span=SMOOTH_SPAN, adjust=False).mean()

# Time in hours
time_hours = (df["time_stamp"] - df["time_stamp"].iloc[0]).dt.total_seconds() / 3600
dt_median = np.median(np.diff(time_hours))
if np.isnan(dt_median) or dt_median <= 0:
    dt_median = 1.0  # safe fallback

# ============================================================
# 3. ROBUST ROLLING LINEAR SLOPE (PER SAMPLE → PER HOUR)
# ============================================================
def rolling_slope(series, win):
    arr = series.values
    n = len(arr)
    slopes = np.full(n, np.nan)
    half = win // 2

    for i in range(n):
        i0 = max(0, i - half)
        i1 = min(n, i + half + 1)
        seg = arr[i0:i1]

        if len(seg) < max(6, win // 2):
            continue

        x = np.arange(len(seg))
        x_mean = x.mean()
        y_mean = seg.mean()

        denom = ((x - x_mean) ** 2).sum()
        if denom == 0:
            continue

        slope_local = ((x - x_mean) * (seg - y_mean)).sum() / denom
        slopes[i] = slope_local

    return slopes


raw_slope_per_sample = rolling_slope(health_smooth, ROLL_WIN)
slope_per_hour = raw_slope_per_sample / dt_median

slope_series = pd.Series(slope_per_hour).fillna(0.0)

# Enforce physical degradation direction (health must not improve)
slope_series[slope_series > 0] = 0.0

# ============================================================
# 4. STABLE REAL-TIME RUL COMPUTATION (SEQUENTIAL)
# ============================================================
rul_list = []
prev_rul = None

for i in range(len(df)):
    h = float(health_smooth.iloc[i])
    s = float(slope_series.iloc[i])

    # If slope is too small → no measurable degradation
    if abs(s) < MIN_SLOPE:
        if prev_rul is None:
            estimated = (h - FAILURE_HEALTH) / (MIN_SLOPE + 1e-9)
            estimated = np.clip(estimated, 1.0, MAX_RUL)
            rul_i = estimated
        else:
            # ✅ Apply slow time-based decay instead of freezing
            rul_i = prev_rul - dt_median * 0.5   # 0.5 hour decay per timestep
    else:
        raw_rul = (h - FAILURE_HEALTH) / (abs(s) + 1e-9)
        raw_rul = np.clip(raw_rul, 0.0, MAX_RUL)
        rul_i = raw_rul

    # Enforce monotonic non-increasing RUL
    if prev_rul is not None:
        rul_i = min(prev_rul, rul_i)

    rul_list.append(rul_i)
    prev_rul = rul_i

rul_series = pd.Series(rul_list)

# ============================================================
# 5. FINAL SMOOTHING & SAFETY CLIPS
# ============================================================
rul_series = rul_series.rolling(
    MEDIAN_SMOOTH_RUL, min_periods=1, center=True
).median()

rul_series = rul_series.clip(0.0, MAX_RUL)
rul_series = rul_series.fillna(method="ffill").fillna(MAX_RUL)

# ============================================================
# 6. SAVE OUTPUT
# ============================================================
out = pd.DataFrame({
    "timestamp": df["time_stamp"],
    "health_index": health_smooth,
    "health_slope_per_hour": slope_series,
    "RealTime_RUL_hours": rul_series
})

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
out.to_csv(OUT_PATH, index=False)

print("✅ Robust Real-Time RUL generated")
print("📁 Saved to:", OUT_PATH)
print("✅ RUL range (hours):", rul_series.min(), "to", rul_series.max())


In [ ]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model

# -------------------------------
# PATHS
# -------------------------------
DATA_PATH = "data/processed/44_processed.csv"
MODEL_PATH = "models/rul_lstm_model.h5"
SCALER_PATH = "models/rul_scaler.pkl"
OUT_PATH = "data/processed/rul_predictions.csv"

SEQUENCE_LENGTH = 30

# -------------------------------
# LOAD MODEL & SCALER
# -------------------------------
model = load_model(MODEL_PATH, compile=False)
scaler = joblib.load(SCALER_PATH)

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv(DATA_PATH)

# -------------------------------
# DROP NON-NUMERICAL META COLUMNS
# -------------------------------
META_COLS = ["timestamp", "time_stamp", "asset_id", "id"]

SENSOR_COLS = [c for c in df.columns if c not in META_COLS]

df_sensors = df[SENSOR_COLS]

# -------------------------------
# SCALE (MATCH TRAINING)
# -------------------------------
X_scaled = scaler.transform(df_sensors.values)

# -------------------------------
# BUILD SEQUENCES
# -------------------------------
X_seq = []

for i in range(len(X_scaled) - SEQUENCE_LENGTH):
    X_seq.append(X_scaled[i:i + SEQUENCE_LENGTH])

X_seq = np.array(X_seq)

# -------------------------------
# PREDICT RUL
# -------------------------------
rul_preds = model.predict(X_seq, verbose=1).flatten()

# -------------------------------
# ALIGN PREDICTIONS TO TIMESTAMPS
# -------------------------------
df_out = df.iloc[SEQUENCE_LENGTH:].copy()
df_out["Predicted_RUL"] = rul_preds

# -------------------------------
# SAVE OUTPUT
# -------------------------------
df_out.to_csv(OUT_PATH, index=False)

print("✅ RUL prediction completed successfully!")
print("📁 Saved to:", OUT_PATH)
print("✅ Total predictions:", len(df_out))


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

DATA_PATH = "data/processed/rul_labeled.csv"
MODEL_OUT = "models/rul_lstm_model.h5"

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv(DATA_PATH)

# -------------------------------
# REMOVE META COLUMNS SAFELY
# -------------------------------
META_COLS = ["timestamp", "time_stamp", "asset_id", "id", "RUL"]

SENSOR_COLS = [c for c in df.columns if c not in META_COLS]

X_raw = df[SENSOR_COLS]
y = df["RUL"].values

# -------------------------------
# SCALE FEATURES (VECTOR SAFE)
# -------------------------------
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save scaler
os.makedirs("models", exist_ok=True)
pd.to_pickle(scaler, "models/rul_scaler.pkl")

# -------------------------------
# SEQUENCE GENERATION (LSTM)
# -------------------------------
SEQUENCE_LENGTH = 30

X_seq = []
y_seq = []

for i in range(len(X_scaled) - SEQUENCE_LENGTH):
    X_seq.append(X_scaled[i:i+SEQUENCE_LENGTH])
    y_seq.append(y[i+SEQUENCE_LENGTH])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# -------------------------------
# TRAIN / VALIDATION SPLIT
# -------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y_seq, test_size=0.2, shuffle=False
)

# -------------------------------
# BUILD LSTM MODEL
# -------------------------------
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_seq.shape[1], X_seq.shape[2])),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# -------------------------------
# TRAIN
# -------------------------------
early_stop = EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# -------------------------------
# SAVE MODEL
# -------------------------------
model.save(MODEL_OUT)

print("✅ RUL LSTM model trained successfully!")
print("📁 Model saved to:", MODEL_OUT)
print("✅ Training samples:", X_train.shape[0])
print("✅ Validation samples:", X_val.shape[0])


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import joblib
import matplotlib.pyplot as plt
import os

PROCESSED_PATH = "data/processed/44_processed.csv"
MODEL_PATH = "models/autoencoder.h5"


def main():
    print("Loading processed data...")
    df = pd.read_csv(PROCESSED_PATH, index_col=0)
    X = df.values

    # Train / validation split (time-based)
    split = int(0.8 * len(X))
    X_train, X_val = X[:split], X[split:]

    input_dim = X.shape[1]
    latent_dim = max(4, input_dim // 4)

    print("Building autoencoder...")
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dense(64, activation="relu")(x)
    latent = layers.Dense(latent_dim, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(latent)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(input_dim, activation="linear")(x)

    autoencoder = models.Model(inputs, outputs)
    autoencoder.compile(optimizer="adam", loss="mse")

    print("Training...")
    history = autoencoder.fit(
        X_train, X_train,
        validation_data=(X_val, X_val),
        epochs=25,
        batch_size=256,
        shuffle=True
    )

    os.makedirs("models", exist_ok=True)
    autoencoder.save(MODEL_PATH)

    print("✅ Model saved to", MODEL_PATH)

    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.legend()
    plt.show()


if __name__ == "__main__":
    main()


In [ ]:
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[codz]
*$py.class

# C extensions
*.so

# CSV files
*.csv

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# PyInstaller
#  Usually these files are written by a python script from a template
#  before PyInstaller builds the exe, so as to inject date/other infos into it.
*.manifest
*.spec

# Installer logs
pip-log.txt
pip-delete-this-directory.txt

# Unit test / coverage reports
htmlcov/
.tox/
.nox/
.coverage
.coverage.*
.cache
nosetests.xml
coverage.xml
*.cover
*.py.cover
.hypothesis/
.pytest_cache/
cover/

# Translations
*.mo
*.pot

# Django stuff:
*.log
local_settings.py
db.sqlite3
db.sqlite3-journal

# Flask stuff:
instance/
.webassets-cache

# Scrapy stuff:
.scrapy

# Sphinx documentation
docs/_build/

# PyBuilder
.pybuilder/
target/

# Jupyter Notebook
.ipynb_checkpoints

# IPython
profile_default/
ipython_config.py

# pyenv
#   For a library or package, you might want to ignore these files since the code is
#   intended to run in multiple environments; otherwise, check them in:
# .python-version

# pipenv
#   According to pypa/pipenv#598, it is recommended to include Pipfile.lock in version control.
#   However, in case of collaboration, if having platform-specific dependencies or dependencies
#   having no cross-platform support, pipenv may install dependencies that don't work, or not
#   install all needed dependencies.
#Pipfile.lock

# UV
#   Similar to Pipfile.lock, it is generally recommended to include uv.lock in version control.
#   This is especially recommended for binary packages to ensure reproducibility, and is more
#   commonly ignored for libraries.
#uv.lock

# poetry
#   Similar to Pipfile.lock, it is generally recommended to include poetry.lock in version control.
#   This is especially recommended for binary packages to ensure reproducibility, and is more
#   commonly ignored for libraries.
#   https://python-poetry.org/docs/basic-usage/#commit-your-poetrylock-file-to-version-control
#poetry.lock
#poetry.toml

# pdm
#   Similar to Pipfile.lock, it is generally recommended to include pdm.lock in version control.
#   pdm recommends including project-wide configuration in pdm.toml, but excluding .pdm-python.
#   https://pdm-project.org/en/latest/usage/project/#working-with-version-control
#pdm.lock
#pdm.toml
.pdm-python
.pdm-build/

# pixi
#   Similar to Pipfile.lock, it is generally recommended to include pixi.lock in version control.
#pixi.lock
#   Pixi creates a virtual environment in the .pixi directory, just like venv module creates one
#   in the .venv directory. It is recommended not to include this directory in version control.
.pixi

# PEP 582; used by e.g. github.com/David-OConnor/pyflow and github.com/pdm-project/pdm
__pypackages__/

# Celery stuff
celerybeat-schedule
celerybeat.pid

# SageMath parsed files
*.sage.py

# Environments
.env
.envrc
.venv
env/
venv/
ENV/
env.bak/
venv.bak/

# Spyder project settings
.spyderproject
.spyproject

# Rope project settings
.ropeproject

# mkdocs documentation
/site

# mypy
.mypy_cache/
.dmypy.json
dmypy.json

# Pyre type checker
.pyre/

# pytype static type analyzer
.pytype/

# Cython debug symbols
cython_debug/

# PyCharm
#  JetBrains specific template is maintained in a separate JetBrains.gitignore that can
#  be found at https://github.com/github/gitignore/blob/main/Global/JetBrains.gitignore
#  and can be added to the global gitignore or merged into this file.  For a more nuclear
#  option (not recommended) you can uncomment the following to ignore the entire idea folder.
#.idea/

# Abstra
# Abstra is an AI-powered process automation framework.
# Ignore directories containing user credentials, local state, and settings.
# Learn more at https://abstra.io/docs
.abstra/

# Visual Studio Code
#  Visual Studio Code specific template is maintained in a separate VisualStudioCode.gitignore
#  that can be found at https://github.com/github/gitignore/blob/main/Global/VisualStudioCode.gitignore
#  and can be added to the global gitignore or merged into this file. However, if you prefer,
#  you could uncomment the following to ignore the entire vscode folder
# .vscode/

# Ruff stuff:
.ruff_cache/

# PyPI configuration file
.pypirc

# Cursor
#  Cursor is an AI-powered code editor. `.cursorignore` specifies files/directories to
#  exclude from AI features like autocomplete and code analysis. Recommended for sensitive data
#  refer to https://docs.cursor.com/context/ignore-files
.cursorignore
.cursorindexingignore

# Marimo
marimo/_static/
marimo/_lsp/
__marimo__/


In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import os

# ==============================
# PAGE CONFIG
# ==============================
st.set_page_config(
    page_title="Offshore Wind Turbine Digital Twin",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("🌊 Offshore Wind Turbine – Digital Twin Dashboard")

# ==============================
# DATA PATHS
# ==============================
ANOMALY_PATH = "data/processed/anomaly_with_root_cause.csv"
HEALTH_PATH = "data/processed/health_index.csv"
RUL_PATH = "data/processed/realtime_rul.csv"

# ==============================
# SAFE TIMESTAMP HANDLER
# ==============================
def normalize_timestamp(df):
    if "time_stamp" in df.columns:
        df["time_stamp"] = pd.to_datetime(df["time_stamp"], errors="coerce")
        return df, "time_stamp"
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.rename(columns={"timestamp": "time_stamp"})
        return df, "time_stamp"
    else:
        raise ValueError(f"No timestamp column found. Columns: {df.columns.tolist()}")

# ==============================
# LOAD DATA
# ==============================
@st.cache_data
def load_data():
    anomaly_df = pd.read_csv(ANOMALY_PATH)
    health_df = pd.read_csv(HEALTH_PATH)
    rul_df = pd.read_csv(RUL_PATH)

    anomaly_df, _ = normalize_timestamp(anomaly_df)
    health_df, _ = normalize_timestamp(health_df)
    rul_df, _ = normalize_timestamp(rul_df)

    return anomaly_df, health_df, rul_df


if not (os.path.exists(ANOMALY_PATH) and os.path.exists(HEALTH_PATH) and os.path.exists(RUL_PATH)):
    st.error("❌ One or more processed data files are missing.")
    st.stop()

anomaly_df, health_df, rul_df = load_data()

# Drop rows with invalid timestamps
anomaly_df = anomaly_df.dropna(subset=["time_stamp"])
health_df = health_df.dropna(subset=["time_stamp"])
rul_df = rul_df.dropna(subset=["time_stamp"])

# ==============================
# SIDEBAR CONTROLS
# ==============================
st.sidebar.header("⚙️ Controls")

start_date = st.sidebar.date_input(
    "Start Date", value=health_df["time_stamp"].min().date()
)
end_date = st.sidebar.date_input(
    "End Date", value=health_df["time_stamp"].max().date()
)

health_df_f = health_df[
    (health_df["time_stamp"].dt.date >= start_date) &
    (health_df["time_stamp"].dt.date <= end_date)
]

rul_df_f = rul_df[
    (rul_df["time_stamp"].dt.date >= start_date) &
    (rul_df["time_stamp"].dt.date <= end_date)
]

anomaly_df_f = anomaly_df[
    (anomaly_df["time_stamp"].dt.date >= start_date) &
    (anomaly_df["time_stamp"].dt.date <= end_date)
]

# ==============================
# KPI METRICS (SAFE)
# ==============================
latest_health = health_df_f["health_index"].dropna().iloc[-1] if not health_df_f.empty else 0
latest_rul = rul_df_f["RealTime_RUL_hours"].dropna().iloc[-1] if not rul_df_f.empty else 0
active_faults = anomaly_df_f["is_anomaly"].sum() if "is_anomaly" in anomaly_df_f else len(anomaly_df_f)

risk_level = "LOW"
if latest_rul < 100:
    risk_level = "CRITICAL"
elif latest_rul < 250:
    risk_level = "HIGH"
elif latest_rul < 400:
    risk_level = "MEDIUM"

col1, col2, col3, col4 = st.columns(4)
col1.metric("⚡ Turbine Health", f"{latest_health:.3f}")
col2.metric("⏳ Real-Time RUL (hrs)", f"{latest_rul:.1f}")
col3.metric("🚨 Active Anomalies", int(active_faults))
col4.metric("⚠️ Risk Level", risk_level)

st.divider()

# ==============================
# HEALTH TREND
# ==============================
st.subheader("📈 Health Degradation Trend")

fig_health = px.line(
    health_df_f,
    x="time_stamp",
    y="health_index",
    title="Health Index Over Time"
)

st.plotly_chart(fig_health, use_container_width=True)

# ==============================
# RUL TREND
# ==============================
st.subheader("⏳ Real-Time Remaining Useful Life")

fig_rul = px.line(
    rul_df_f,
    x="time_stamp",
    y="RealTime_RUL_hours",
    title="Real-Time RUL (Hours)"
)

st.plotly_chart(fig_rul, use_container_width=True)

# ==============================
# FAULT & RCA ANALYSIS
# ==============================
st.subheader("🛠 Fault Distribution by Subsystem (RCA)")

if "root_cause_physical" in anomaly_df_f.columns:
    rca_counts = anomaly_df_f["root_cause_physical"].value_counts().reset_index()
    rca_counts.columns = ["Subsystem", "Fault Count"]

    fig_rca = px.bar(
        rca_counts,
        x="Subsystem",
        y="Fault Count",
        title="Fault Count per Subsystem"
    )

    st.plotly_chart(fig_rca, use_container_width=True)
else:
    st.warning("RCA column not found in anomaly file.")

# ==============================
# ANOMALY TABLE
# ==============================
st.subheader("📋 Recent Anomalies")

display_cols = [c for c in anomaly_df_f.columns if "Unnamed" not in c]
st.dataframe(anomaly_df_f[display_cols].tail(20), use_container_width=True)

# ==============================
# FOOTER
# ==============================
st.markdown("---")
st.markdown(
    "✅ **Digital Twin includes:** Anomaly Detection, Root Cause Analysis, "
    "Health Index Estimation, and Real-Time RUL Prediction."
)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/health_index.csv")
df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df = df.sort_values('time_stamp')

# smoothed health (same as in pipeline)
health_sm = df['health_index'].astype(float).ewm(span=30).mean()
dt_hours = (df['time_stamp'].diff().dt.total_seconds().median()) / 3600.0

# approx slope (per sample)
slope = np.gradient(health_sm, dt_hours)
print("slope stats:", pd.Series(slope).describe())

# count tiny slopes
eps = 1e-4
print("tiny slope count:", (np.abs(slope) < eps).sum())

# compute raw_rul before clipping for inspection
failure_h = 0.05
raw_rul = (health_sm - failure_h) / (np.abs(slope) + 1e-12)
print("raw_rul stats:", raw_rul.describe())
print("count > 2000:", (raw_rul > 2000).sum())
